# S3.19 - try/except for bad records
**Date completed:** October 2026  
**Status:** In Progress  
**Key concept:** Spark is lazy. Data errors appear at the action, so the try must wrap the action.

In [0]:
# ============================================================
# S3.19 — Cell 2
# Error Handling, Lazy Evaluation and Safe Casting
#
# Goal:
# 1. Understand that Spark transformations are lazy.
# 2. See that execution errors surface when an action runs.
# 3. Compare cast() and try_cast() under ANSI behaviour.
# 4. Distinguish source NULL from malformed input.
# 5. Identify and quarantine bad records.
# 6. Keep valid rows available for downstream processing.
# 7. Understand where try/except belongs in Spark workflows.
#
# Environment:
# Databricks Serverless / Spark Connect
#
# Important:
# Creating a transformation does NOT prove that the data is valid.
#
# The transformation plan is executed only when an action requires
# Spark to produce a result.
# ============================================================

import pyspark.sql.functions as F

print("=== S3.19: ERROR HANDLING AND SAFE CASTING ===")
print()


# ------------------------------------------------------------
# Step 1 — Inspect ANSI Mode
#
# ANSI mode affects how some invalid SQL operations behave.
#
# For example:
#
# CAST('abc' AS INT)
#
# Under ANSI-enabled behaviour:
# → malformed conversion raises an error during execution.
#
# try_cast():
# → returns NULL for a supported malformed conversion.
# ------------------------------------------------------------

ansi_mode = spark.conf.get(
    "spark.sql.ansi.enabled")

print(
    f"Spark ANSI mode: {ansi_mode}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 2 — Controlled Source Data
#
# Cases:
#
# "100" → valid integer string
# "250" → valid integer string
# "abc" → malformed integer string
# NULL  → genuine source NULL
# "75"  → valid integer string
# ------------------------------------------------------------

data = [
    (1, "100"),
    (2, "250"),
    (3, "abc"),
    (4, None),
    (5, "75")
]


df = spark.createDataFrame(
    data,
    [
        "id",
        "x"
    ]
)

print("=== STEP 2: SOURCE DATA ===")

df.show(
    truncate=False
)

print()

In [0]:
# ------------------------------------------------------------
# Step 3 — Transformation Only
#
# withColumn() is a transformation.
#
# Spark builds the logical execution plan but does not need to
# evaluate every row yet.
#
# Therefore malformed data may remain undiscovered at this stage.
# ------------------------------------------------------------

print(
    "=== STEP 3: TRANSFORMATION ONLY ==="
)


df_cast = df.withColumn(
    "n",
    F.col(
        "x"
    ).cast(
        "int"
    ))

print("""
df_cast was created successfully.

This does NOT mean every value can actually be cast.

Spark has built the transformation plan, but the rows have
not necessarily been evaluated yet.
""")

In [0]:
# ------------------------------------------------------------
# Step 4 — Action Inside try/except
#
# show() is an action.
#
# Spark now needs to execute the transformation plan.
#
# If ANSI mode is enabled, the value:
#
# "abc"
#
# cannot be converted to INT using normal cast().
#
# The exception therefore surfaces during execution.
# ------------------------------------------------------------

print(
    "=== STEP 4: ACTION INSIDE try/except ==="
)

try:

    df_cast.show(truncate=False)

    print("cast() completed without an exception.")

except Exception as e:

    print("Spark execution failed during the action.")

    print(f"Exception type: {type(e).__name__}")

    # Keep the displayed error concise for the notebook.
    print(f"Message preview: {str(e)[:300]}")

print()

In [0]:
# ------------------------------------------------------------
# Step 5 — Why the Error Appears at the Action
#
# Conceptually:
#
# withColumn(cast)
#       ↓
# Build execution plan
#       ↓
# No full row evaluation yet
#
# show()
#       ↓
# Spark executes plan
#       ↓
# Reads "abc"
#       ↓
# Attempts INT conversion
#       ↓
# Error under ANSI-enabled cast behaviour
# ------------------------------------------------------------

print(
    "=== STEP 5: LAZY EVALUATION ==="
)


print("""
Transformation:

df.withColumn(...)
→ builds a new DataFrame / execution plan.


Actions such as:

show()
count()
collect()
take()
first()

→ request execution/results.


Therefore:

A data-related execution error can appear at an action,
even though the problematic transformation was defined earlier.
""")

In [0]:
# ------------------------------------------------------------
# Step 6 — Safe Conversion with try_cast()
#
# try_cast():
#
# valid conversion
# → converted value
#
# malformed supported conversion
# → NULL
#
# source NULL
# → NULL
#
# This keeps the Spark query running.
#
# IMPORTANT:
# try_cast() does not magically make the record valid.
# NULL results still need DQ validation.
# ------------------------------------------------------------

print(
    "=== STEP 6: try_cast() ==="
)


df_safe = df.withColumn(

    "n",

    F.expr("try_cast(x AS INT)"))


df_safe.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 7 — Expected Conversion Behaviour
# ------------------------------------------------------------

print(
    "=== STEP 7: CONVERSION INTERPRETATION ==="
)


print("""
Expected:

x = "100"
→ n = 100

x = "250"
→ n = 250

x = "abc"
→ n = NULL
→ malformed conversion

x = NULL
→ n = NULL
→ genuine source NULL

x = "75"
→ n = 75


Important:

These two cases both produce:

n = NULL

but they do NOT mean the same thing.

Source NULL:
x IS NULL

Bad conversion:
x IS NOT NULL
AND
n IS NULL
""")

In [0]:
# ------------------------------------------------------------
# Step 8 — Identify Bad Conversion Records
#
# Bad conversion rule:
#
# Source value exists
# AND
# parsed value is NULL
#
# Therefore:
#
# x IS NOT NULL
# AND
# n IS NULL
#
# This correctly excludes genuine source NULL values.
# ------------------------------------------------------------

print(
    "=== STEP 8: IDENTIFY BAD RECORDS ==="
)


df_bad = (

    df_safe

    .filter(

        F.col("x").isNotNull() & F.col("n").isNull()))


bad_count = (
    df_bad.count()
)


print(
    f"Bad conversion records: {bad_count}"
)


df_bad.show(
    truncate=False
)

assert bad_count == 1, (
    "Expected exactly one malformed conversion record"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 9 — Identify Valid Records
#
# There are two possible business interpretations:
#
# A. Valid parsed rows only:
#       n IS NOT NULL
#
# B. Keep genuine NULL source values as valid nullable records:
#       source NULL may remain in the Silver flow
#
# Which rule is correct depends on the business contract.
#
# For this exercise:
#
# good conversion
# → n IS NOT NULL
# ------------------------------------------------------------

print(
    "=== STEP 9: VALID CONVERSIONS ==="
)


df_good = (

    df_safe

    .filter(F.col("n").isNotNull()))

good_count = (df_good.count())


print(
    f"Successfully converted records: {good_count}"
)


df_good.show(truncate=False)

assert good_count == 3, (
    "Expected three successful conversions"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 10 — Genuine Source NULLs
#
# Genuine NULL input is different from malformed data.
# ------------------------------------------------------------

print(
    "=== STEP 10: SOURCE NULL RECORDS ==="
)


df_source_null = (

    df_safe
    .filter(
        F.col(
            "x"
        ).isNull()))

source_null_count = (
    df_source_null.count())

print(
    f"Source NULL records: {source_null_count}")

df_source_null.show(
    truncate=False)

assert source_null_count == 1, (
    "Expected one genuine source NULL"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 11 — Add Data-Quality Status
#
# Instead of only filtering records, we can classify every row.
#
# This makes the DQ decision visible and auditable.
# ------------------------------------------------------------

print(
    "=== STEP 11: DATA-QUALITY CLASSIFICATION ==="
)


df_classified = (

    df_safe

    .withColumn(

        "dq_status",

        F.when(
            F.col("x").isNull(),
            "SOURCE_NULL"
        )

        .when(
            F.col("n").isNull(),
            "INVALID_INTEGER"
        )

        .otherwise(
            "VALID"
        )
    )
)


df_classified.show(
    truncate=False
)

print()

In [0]:
# ------------------------------------------------------------
# Step 12 — DQ Distribution
# ------------------------------------------------------------

print(
    "=== STEP 12: DQ DISTRIBUTION ==="
)


(
    df_classified

    .groupBy(
        "dq_status"
    )

    .count()

    .orderBy(
        "dq_status"
    )

    .show(
        truncate=False
    )
)

print()

In [0]:
# ------------------------------------------------------------
# Step 13 — Silver / Quarantine Pattern
#
# Common enterprise flow:
#
# Raw value
#    ↓
# Safe parse
#    ↓
# classify result
#    ↓
# ┌───────────────────┐
# │                   │
# VALID            INVALID
# │                   │
# Silver           Quarantine
#
#
# Genuine source NULL handling depends on whether the
# target column is allowed to be NULL.
# ------------------------------------------------------------

print(
    "=== STEP 13: SILVER / QUARANTINE PATTERN ==="
)


df_quarantine = (

    df_classified

    .filter(
        F.col(
            "dq_status"
        ) == "INVALID_INTEGER"
    )

    .select(
        "id",
        "x",
        "n",
        "dq_status"
    )
)


print("\nQuarantine records:")

df_quarantine.show(
    truncate=False
)


df_silver_ready = (

    df_classified

    .filter(
        F.col(
            "dq_status"
        ) != "INVALID_INTEGER"
    )
)


print("\nRecords allowed to continue:")

df_silver_ready.show(
    truncate=False
)

print()

In [0]:
# ------------------------------------------------------------
# Step 14 — Reconciliation
#
# Every source record should be classified exactly once.
#
# Source:
#
# VALID conversions
# +
# genuine source NULLs
# +
# malformed conversions
#
# =
# total source rows
# ------------------------------------------------------------

print(
    "=== STEP 14: RECONCILIATION ==="
)


source_count = (
    df.count()
)


classified_count = (
    df_classified.count()
)


print(
    f"Source rows     : {source_count}"
)

print(
    f"Classified rows : {classified_count}"
)


assert source_count == classified_count, (
    "Row reconciliation failed"
)


assert (
    good_count
    +
    source_null_count
    +
    bad_count
    ==
    source_count
), "DQ category reconciliation failed"


print(
    "DQ reconciliation passed."
)

print()

In [0]:
# ------------------------------------------------------------
# Step 15 — try/except Does NOT Replace DQ Logic
#
# try/except:
#
# protects the notebook/application from an execution exception.
#
# try_cast + validation:
#
# gives us record-level control over malformed input.
#
# These solve different problems.
# ------------------------------------------------------------

print(
    "=== STEP 15: try/except vs try_cast() ==="
)


print("""
try/except:

→ Handles an execution exception.
→ Useful for operational error handling.
→ Works around the action that may fail.


try_cast():

→ Handles malformed conversion at the row/value level.
→ Produces NULL instead of failing the supported conversion.
→ Allows bad rows to be identified and quarantined.


They are NOT substitutes for each other.
""")

In [0]:
# ------------------------------------------------------------
# Step 16 — Production Error-Handling Principle
#
# This learning example uses:
#
# except Exception
#
# because we want to demonstrate where Spark raises the error.
#
# In production:
#
# - log enough context
# - capture pipeline/run identifiers
# - classify expected failures where practical
# - do not silently swallow serious exceptions
# - re-raise when the pipeline should fail
#
# A caught exception does not automatically mean the pipeline
# should continue successfully.
# ------------------------------------------------------------

print(
    "=== STEP 16: PRODUCTION PRINCIPLE ==="
)


print("""
Learning demo:

try:
    action()
except Exception as e:
    print(...)


Production principle:

try:
    action()

except expected_exception:
    log useful context
    handle according to business/operational rule

except Exception:
    log unexpected failure
    raise


Do not blindly catch every exception and pretend
the pipeline succeeded.
""")

In [0]:
# ------------------------------------------------------------
# Step 17 — Repeated Actions
#
# Each action can trigger Spark execution.
#
# Example:
#
# df_bad.count()
# df_bad.show()
#
# are two separate actions.
#
# In learning notebooks this is fine because we want to inspect
# the output.
#
# In production pipelines avoid unnecessary repeated actions.
# ------------------------------------------------------------

print(
    "=== STEP 17: ACTION AWARENESS ==="
)


print("""
Actions used in this lesson:

show()
count()


Important:

A transformation is lazy.

Each action can trigger execution of the required lineage.

Therefore avoid unnecessary actions simply for debugging
in production-scale pipelines.
""")

In [0]:
# ------------------------------------------------------------
# Step 18 — Final Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
LAZY EVALUATION:

withColumn(cast)
→ builds transformation plan

action
→ executes plan
→ data error may surface here


cast():

Malformed conversion behaviour depends on Spark settings.

Under ANSI-enabled behaviour:

invalid conversion
→ execution error


try_cast():

valid value
→ converted value

malformed supported value
→ NULL

source NULL
→ NULL


BAD-RECORD DETECTION:

source IS NOT NULL
AND
parsed_value IS NULL

→ malformed conversion


IMPORTANT:

source NULL
≠
bad conversion


DQ PATTERN:

Raw value
    ↓
try_cast()
    ↓
classify
    ↓
VALID / SOURCE_NULL / INVALID
    ↓
Silver / business handling / quarantine


try/except:

→ execution-level exception handling


try_cast():

→ row/value-level safe conversion


CORE PRINCIPLE:

Do not rely on try/except alone to manage bad records.

Prefer:

safe conversion
      ↓
explicit validation
      ↓
classification
      ↓
quarantine / business handling
      ↓
reconciliation
""")

print("=== S3.19 CELL 2 COMPLETED ===")

In [0]:
# ============================================================
# Cell 3 - Checkpoint: the three results in one output
#
# Goal:
# 1. ANSI mode and whether cast() fails at the action
# 2. try_cast() values for each row
# 3. DQ status counts
#
# Only 5 rows, so collect() is safe here.
# ============================================================

import pyspark.sql.functions as F

print("ANSI mode:", spark.conf.get("spark.sql.ansi.enabled"))

data = [(1, "100"), (2, "250"), (3, "abc"), (4, None), (5, "75")]
df = spark.createDataFrame(data, ["id", "x"])

# ------------------------------------------------------------
# Step 1 - Does cast() raise an error at the action?
# ------------------------------------------------------------
try:
    df.withColumn("n", F.col("x").cast("int")).collect()
    print("cast(): completed without an exception")
except Exception as e:
    print("cast(): exception ->", type(e).__name__)
    print(str(e)[:150])

# ------------------------------------------------------------
# Step 2 - try_cast() values: (id, x, n)
# ------------------------------------------------------------
df_safe = df.withColumn("n", F.expr("try_cast(x AS INT)"))
print("try_cast rows:", [(r["id"], r["x"], r["n"]) for r in df_safe.collect()])

# ------------------------------------------------------------
# Step 3 - DQ status counts
# ------------------------------------------------------------
df_dq = df_safe.withColumn(
    "dq_status",
    F.when(F.col("x").isNull(), "SOURCE_NULL")
     .when(F.col("n").isNull(), "INVALID_INTEGER")
     .otherwise("VALID")
)
print("DQ counts:", {r["dq_status"]: r["count"]
                     for r in df_dq.groupBy("dq_status").count().collect()})

# S3.19 — Key Takeaways: try/except for Bad Records

## 1. Lazy Evaluation and Errors

- `withColumn(cast)` is a transformation: Spark builds the execution plan.
- A malformed value may not fail while the transformation is being defined.
- The error appears when Spark executes a path that actually evaluates the bad cast.
- Actions such as `show()` or `collect()` on the cast column can therefore expose the error.

### Important

Do not assume every action will necessarily evaluate every expression.

For example:

```python
df_cast.count()
```

may allow Spark to avoid evaluating an unused cast column because of optimizer column pruning.

Therefore the precise rule is:

```text
Bad expression
      ↓
Lazy transformation
      ↓
Action requires that expression
      ↓
Expression is evaluated
      ↓
Execution error surfaces
```

For lazy data-related failures, `try/except` should wrap the action that materializes the problematic expression.

---

## 2. cast() vs try_cast()

### cast()

In our workspace:

```text
spark.sql.ansi.enabled = true
```

With ANSI mode enabled, malformed conversions such as:

```text
"abc" → INT
```

raise an execution error when evaluated.

Observed:

```text
NumberFormatException
[CAST_INVALID_INPUT]
```

---

### try_cast()

```python
F.expr(
    "try_cast(x AS INT)"
)
```

For supported conversions:

```text
"100" → 100
"250" → 250
"abc" → NULL
NULL  → NULL
"75"  → 75
```

`try_cast()` prevents malformed conversion values from failing the query.

### Important

```text
try_cast()
→ safe conversion

NOT

try_cast()
→ valid data
```

A resulting NULL still requires validation.

---

## 3. Source NULL vs Bad Conversion

These two cases both produce:

```text
n = NULL
```

but have different meanings.

### Genuine Source NULL

```text
x IS NULL
```

Example:

```text
x = NULL
n = NULL
```

This may be valid if the business column is nullable.

---

### Failed Conversion

```text
x IS NOT NULL
AND
n IS NULL
```

Example:

```text
x = "abc"
n = NULL
```

This indicates that a source value existed but could not be converted.

### Bad-record rule

```python
F.col("x").isNotNull() & F.col("n").isNull()
```

This avoids incorrectly classifying genuine source NULLs as failed casts.

---

## 4. DQ Classification

Our classification:

```text
VALID
SOURCE_NULL
INVALID_INTEGER
```

Example logic:

```python
F.when(
    F.col("x").isNull(),
    "SOURCE_NULL"
)
.when(
    F.col("n").isNull(),
    "INVALID_INTEGER"
)
.otherwise(
    "VALID"
)
```

Conceptually:

```text
Raw value
    ↓
try_cast()
    ↓
Classify
    ↓
┌──────────────┬──────────────┬─────────────────┐
VALID          SOURCE_NULL    INVALID_INTEGER
↓              ↓              ↓
Silver /       Business       Quarantine /
continue       rule           reject handling
```

The correct treatment of `SOURCE_NULL` depends on the target-column business contract.

---

## 5. Reconciliation

Every source row should be accounted for exactly once.

```text
VALID
+
SOURCE_NULL
+
INVALID_INTEGER
=
SOURCE ROWS
```

Observed:

```text
VALID           = 3
INVALID_INTEGER = 1
SOURCE_NULL     = 1
-------------------
TOTAL           = 5
```

Therefore:

```text
3 + 1 + 1 = 5
```

Reconciliation passed.

---

## 6. try/except vs try_cast()

These solve different problems.

### try/except

```text
Execution / operational error handling
```

Example:

```python
try:
    df_cast.show()

except Exception as e:
    ...
```

It handles the exception produced while Spark executes the operation.

---

### try_cast() + Validation

```text
Row-level data-quality handling
```

Example:

```text
"abc"
  ↓
try_cast()
  ↓
NULL
  ↓
DQ rule
  ↓
INVALID_INTEGER
  ↓
Quarantine
```

### Easy way to remember

```text
try/except
→ Did the Spark operation fail?

try_cast()
→ Did this individual value fail conversion?
```

---

## 7. Production Error Handling

For the learning exercise:

```python
except Exception as e:
    print(...)
```

is useful for demonstrating the failure.

In production:

- Log useful execution context.
- Include pipeline/run identifiers where available.
- Handle expected failures according to defined rules.
- Do not silently swallow unexpected exceptions.
- Re-raise unexpected failures when the pipeline should fail.

Conceptually:

```python
try:
    action()

except expected_exception:
    log_context()
    handle_expected_failure()

except Exception:
    log_unexpected_failure()
    raise
```

Catching an exception does not automatically mean the pipeline should continue successfully.

---

## 8. Transformations vs Actions

### Transformation

```python
df.withColumn(
    "n",
    F.col("x").cast("int")
)
```

Builds a DataFrame execution plan.

---

### Actions

Examples:

```text
show()
collect()
take()
first()
count()
```

Actions request execution/results.

### Important nuance

An action only exposes a bad expression if Spark actually needs to evaluate that expression for the requested result.

Therefore:

```text
Action
≠
guaranteed evaluation of every DataFrame column
```

Spark's optimizer can prune unused columns and expressions.

---

## 9. Observed in Our Workspace

### ANSI mode

```text
true
```

### Normal cast

Input:

```text
"abc"
```

Observed result:

```text
NumberFormatException
[CAST_INVALID_INPUT]
```

when the malformed cast was evaluated.

### try_cast()

Observed:

```text
"abc" → None
None  → None
```

### DQ counts

```text
VALID           = 3
INVALID_INTEGER = 1
SOURCE_NULL     = 1

TOTAL           = 5
```

---

# 10. Related Interview Questions and Answers

### Q1. Why does the error not occur when `withColumn(cast)` is defined?

Because Spark transformations are lazy. The expression is added to the execution plan and is normally evaluated only when an action requires its result.

---

### Q2. Does every action necessarily evaluate the bad cast?

No. Spark may prune expressions that are not required for the requested result.

For example, a row count may not need to evaluate an unused cast column.

---

### Q3. What is the difference between `cast()` and `try_cast()`?

With ANSI behaviour enabled, a malformed `cast()` can fail execution.

`try_cast()` returns NULL for supported malformed conversions instead, allowing the pipeline to validate the record explicitly.

---

### Q4. Why can't we simply treat every NULL after `try_cast()` as a bad record?

Because the original source value may already have been NULL.

We distinguish:

```text
source NULL:
x IS NULL

bad conversion:
x IS NOT NULL AND parsed_value IS NULL
```

---

### Q5. Does `try_cast()` solve data-quality problems?

No.

It prevents a malformed conversion from failing the query, but the resulting NULL still needs to be classified, validated and handled according to business rules.

---

### Q6. What is the difference between `try/except` and `try_cast()`?

`try/except` operates at execution/error-handling level.

`try_cast()` operates at individual value-conversion level.

---

### Q7. What should happen to malformed records in a Silver-layer pipeline?

A common pattern is:

```text
Raw record
    ↓
Safe parse
    ↓
DQ classification
    ↓
Valid → Silver

Invalid → quarantine / reject handling
```

The exact rule depends on the business data contract.

---

### Q8. Why should unexpected exceptions usually be re-raised?

Because silently catching an unexpected failure can make a failed pipeline appear successful and hide operational problems.

---

## 11. Core Pattern

```text
Raw STRING
    ↓
try_cast()
    ↓
┌───────────────────────────┐
│                           │
Parsed value              NULL
│                           │
VALID                 Check source
                           ↓
                  ┌────────┴─────────┐
                  │                  │
             Source NULL       Source non-NULL
                  │                  │
            Business rule      Bad conversion
                                     ↓
                                 Quarantine
```

## Final Rule

```text
try/except
→ execution-level protection

try_cast()
→ value-level safe conversion

DQ validation
→ determines what the NULL actually means
```

**S3.19 — Error Handling and Safe Casting completed after checkpoint validation.**